# Retail Demand Forecasting — Data Understanding

## Objective

Understand the structure, granularity, data quality, and business meaning of the raw retail transaction data before cleaning or building forecasting models.

## Initial Questions

- What does one row represent?
- How many rows and columns are in the dataset?
- What data types are present?
- Are there missing values?
- Are there duplicates?
- What date range does the dataset cover?
- Are quantities or prices ever zero or negative?
- How many unique products, customers, invoices, and countries are present?
- How should cancellations and returns be handled?

In [6]:
import pandas as pd
import numpy as np

excel_file = pd.ExcelFile("../data/raw/online_retail_II.xlsx")


In [7]:
retail_2009_2010 = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name=0
)

retail_2010_2011 = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name=1
)

print("2009-2010:", retail_2009_2010.shape)
print("2010-2011:", retail_2010_2011.shape)

2009-2010: (525461, 8)
2010-2011: (541910, 8)


In [8]:
print(retail_2009_2010.columns)
print(retail_2010_2011.columns)

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')
Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')


In [9]:
retail_2009_2010.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
retail_2010_2011.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [11]:
print(retail_2009_2010.dtypes)
print()
print(retail_2010_2011.dtypes)

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [12]:
print("2009-2010 Missing Values")
print(retail_2009_2010.isna().sum())

print("\n2010-2011 Missing Values")
print(retail_2010_2011.isna().sum())

2009-2010 Missing Values
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

2010-2011 Missing Values
Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64


In [13]:
print(retail_2009_2010[["Quantity", "Price"]].describe())

print()

print(retail_2010_2011[["Quantity", "Price"]].describe())

            Quantity          Price
count  525461.000000  525461.000000
mean       10.337667       4.688834
std       107.424110     146.126914
min     -9600.000000  -53594.360000
25%         1.000000       1.250000
50%         3.000000       2.100000
75%        10.000000       4.210000
max     19152.000000   25111.090000

            Quantity          Price
count  541910.000000  541910.000000
mean        9.552234       4.611138
std       218.080957      96.759765
min    -80995.000000  -11062.060000
25%         1.000000       1.250000
50%         3.000000       2.080000
75%        10.000000       4.130000
max     80995.000000   38970.000000


In [14]:
print("2009-2010")
print("Negative quantity:", (retail_2009_2010["Quantity"] < 0).sum())
print("Zero quantity:", (retail_2009_2010["Quantity"] == 0).sum())
print("Negative price:", (retail_2009_2010["Price"] < 0).sum())
print("Zero price:", (retail_2009_2010["Price"] == 0).sum())

print("\n2010-2011")
print("Negative quantity:", (retail_2010_2011["Quantity"] < 0).sum())
print("Zero quantity:", (retail_2010_2011["Quantity"] == 0).sum())
print("Negative price:", (retail_2010_2011["Price"] < 0).sum())
print("Zero price:", (retail_2010_2011["Price"] == 0).sum())

2009-2010
Negative quantity: 12326
Zero quantity: 0
Negative price: 3
Zero price: 3687

2010-2011
Negative quantity: 10624
Zero quantity: 0
Negative price: 2
Zero price: 2515


In [15]:
retail_2009_2010[
    retail_2009_2010["Quantity"] < 0
][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(15)

,Invoice,StockCode,Description,Quantity,Price
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25
181,C489449,21896,POTTING SHED TWINE,-6,2.10
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95
183,C489449,21871,SAVE THE PLANET MUG,-12,1.25
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25


In [16]:
retail_2010_2011[
    retail_2010_2011["Quantity"] < 0
][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(15)

,Invoice,StockCode,Description,Quantity,Price
141,C536379,D,Discount,-1,27.50
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65
937,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25


In [17]:
retail_2009_2010[
    retail_2009_2010["Price"] < 0
]

retail_2010_2011[
    retail_2010_2011["Price"] < 0
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [18]:
retail_2009_2010[
    retail_2009_2010["Price"] == 0
][["Invoice", "StockCode", "Description", "Quantity", "Price"]].head(15)

,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,NaN,-50,0.0
3114,489655,20683,NaN,-44,0.0
3161,489659,21350,NaN,230,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
3731,489781,84292,NaN,17,0.0
4296,489806,18010,NaN,-770,0.0


### Initial Cleaning Rules

1. Keep rows with valid `InvoiceDate`, `StockCode`, and `Quantity`.
2. Exclude cancelled/returned transactions where:
   - `Invoice` starts with `C`, or
   - `Quantity` is negative.
3. Exclude rows with `Price < 0` because these represent accounting adjustments rather than product sales.
4. Investigate `Price == 0` separately before deciding whether to retain or exclude.
5. Missing `Customer ID` does not automatically invalidate a transaction for demand forecasting.
6. Missing `Description` does not automatically invalidate a row if `StockCode`, `Quantity`, and `InvoiceDate` remain usable.

In [20]:
for name, df in {
    "2009-2010": retail_2009_2010,
    "2010-2011": retail_2010_2011
}.items():

    cancelled = df["Invoice"].astype(str).str.startswith("C").sum()
    negative_qty = (df["Quantity"] < 0).sum()
    negative_price = (df["Price"] < 0).sum()
    zero_price = (df["Price"] == 0).sum()

    print(name)
    print("Cancelled rows:", cancelled)
    print("Negative quantity:", negative_qty)
    print("Negative price:", negative_price)
    print("Zero price:", zero_price)
    print()

2009-2010
Cancelled rows: 10206
Negative quantity: 12326
Negative price: 3
Zero price: 3687

2010-2011
Cancelled rows: 9288
Negative quantity: 10624
Negative price: 2
Zero price: 2515



In [21]:
cancelled_mask_0910 = (
    retail_2009_2010["Invoice"]
    .astype(str)
    .str.startswith("C")
)

negative_qty_mask_0910 = (
    retail_2009_2010["Quantity"] < 0
)

print(
    "Cancelled + negative quantity:",
    (cancelled_mask_0910 & negative_qty_mask_0910).sum()
)

print(
    "Negative quantity but NOT cancelled:",
    (~cancelled_mask_0910 & negative_qty_mask_0910).sum()
)

Cancelled + negative quantity: 10205
Negative quantity but NOT cancelled: 2121


In [22]:
retail_2009_2010[
    ~cancelled_mask_0910
    & negative_qty_mask_0910
][
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
].head(20)

,Invoice,StockCode,Description,Quantity,Price
263,489464,21733,85123a mixed,-96,0.0
283,489463,71477,short,-240,0.0
284,489467,85123A,21733 mixed,-192,0.0
470,489521,21646,NaN,-50,0.0
3114,489655,20683,NaN,-44,0.0
3162,489660,35956,lost,-1043,0.0
3168,489663,35605A,damages,-117,0.0
4296,489806,18010,NaN,-770,0.0
4538,489820,21133,invcd as 84879?,-720,0.0
4566,489821,85049G,NaN,-240,0.0


In [23]:
retail_2009_2010[
    cancelled_mask_0910
    & ~negative_qty_mask_0910
][
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
]

,Invoice,StockCode,Description,Quantity,Price
76799,C496350,M,Manual,1,373.57


In [24]:
zero_price_positive_qty_0910 = retail_2009_2010[
    (retail_2009_2010["Price"] == 0)
    & (retail_2009_2010["Quantity"] > 0)
]

zero_price_positive_qty_0910[
    ["Invoice", "StockCode", "Description", "Quantity", "Price"]
].head(30)

,Invoice,StockCode,Description,Quantity,Price
3161,489659,21350,NaN,230,0.0
3731,489781,84292,NaN,17,0.0
4674,489825,22076,6 RIBBONS EMPIRE,12,0.0
5904,489861,DOT,DOTCOM POSTAGE,1,0.0
6378,489882,35751C,NaN,12,0.0
6555,489898,79323G,NaN,954,0.0
6581,489903,21166,NaN,48,0.0
6781,489998,48185,DOOR MAT FAIRY CAKE,2,0.0
7204,490015,21982,NaN,467,0.0
9249,490123,84508B,NaN,184,0.0


In [25]:
print("2009-2010 zero-price positive quantity rows:")
print(len(zero_price_positive_qty_0910))

print(
    "Missing Customer ID:",
    zero_price_positive_qty_0910["Customer ID"].isna().sum()
)

print(
    "Missing Description:",
    zero_price_positive_qty_0910["Description"].isna().sum()
)

print(
    "Unique StockCodes:",
    zero_price_positive_qty_0910["StockCode"].nunique()
)

2009-2010 zero-price positive quantity rows:
1566
Missing Customer ID: 1535
Missing Description: 1101
Unique StockCodes: 970


In [26]:
zero_price_positive_qty_0910["Description"].value_counts(dropna=False).head(20)

zero_price_positive_qty_0910["StockCode"].value_counts().head(20)



StockCode
22469     9
79321     8
21116     8
22501     8
22950     8
22197     7
22139     7
46000M    7
84795D    7
84990     7
22734     7
22355     6
20747     6
21523     6
84016     6
22429     6
22502     6
22372     6
21478     5
90002A    5
Name: count, dtype: int64

### Zero-Price Transaction Decision

Positive-quantity transactions with a zero unit price were investigated separately.

In 2009–2010, 1,566 rows had positive quantity but zero price. Approximately 98% had no Customer ID and 70% had no product description. The records were also spread across 970 StockCodes and included manual, postage and stock-adjustment style entries.

Given the small volume and strong evidence that these records primarily represent internal stock movements or incomplete transactions rather than normal customer purchases, zero-price rows will be excluded from the demand forecasting dataset.

This prevents operational stock adjustments from being interpreted as customer demand.

In [27]:
zero_price_positive_qty_1011 = retail_2010_2011[
    (retail_2010_2011["Price"] == 0)
    & (retail_2010_2011["Quantity"] > 0)
]

print("2010-2011 zero-price positive quantity rows:")
print(len(zero_price_positive_qty_1011))

print(
    "Missing Customer ID:",
    zero_price_positive_qty_1011["Customer ID"].isna().sum()
)

print(
    "Missing Description:",
    zero_price_positive_qty_1011["Description"].isna().sum()
)

print(
    "Unique StockCodes:",
    zero_price_positive_qty_1011["StockCode"].nunique()
)

2010-2011 zero-price positive quantity rows:
1179
Missing Customer ID: 1139
Missing Description: 592
Unique StockCodes: 681


### Forecasting Transaction Rule

To represent genuine customer demand, the forecasting dataset will retain only rows where:

- `Quantity > 0`
- `Price > 0`

Negative quantities primarily represent cancellations, returns, damages, shortages, or stock adjustments.

Zero-price transactions were also investigated and were heavily associated with missing customer IDs, missing descriptions, and non-standard operational records. These rows will therefore be excluded from the demand forecasting population.

In [28]:
print("2009-2010 exact duplicates:")
print(retail_2009_2010.duplicated().sum())

print("\n2010-2011 exact duplicates:")
print(retail_2010_2011.duplicated().sum())

2009-2010 exact duplicates:
6865

2010-2011 exact duplicates:
5268


In [29]:
retail_2009_2010[
    retail_2009_2010.duplicated(keep=False)
].sort_values(
    by=["Invoice", "StockCode", "InvoiceDate"]
).head(20)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


### Duplicate Handling

Exact duplicate rows were identified where all transaction fields matched.

These records would artificially inflate unit demand if retained, so exact duplicates will be removed before aggregation and forecasting.

Only exact row-level duplicates will be removed. Repeated purchases of the same product will remain if any transaction-level field differs.

In [31]:
clean_2009_2010 = retail_2009_2010.copy()
clean_2010_2011 = retail_2010_2011.copy()
clean_2009_2010 = clean_2009_2010.drop_duplicates()
clean_2010_2011 = clean_2010_2011.drop_duplicates()

In [33]:
clean_2009_2010 = clean_2009_2010[
    (clean_2009_2010["Quantity"] > 0)
    & (clean_2009_2010["Price"] > 0)
]

clean_2010_2011 = clean_2010_2011[
    (clean_2010_2011["Quantity"] > 0)
    & (clean_2010_2011["Price"] > 0)
]

print("2009-2010")
print("Raw rows:", len(retail_2009_2010))
print("Clean rows:", len(clean_2009_2010))
print("Rows removed:", len(retail_2009_2010) - len(clean_2009_2010))

print("\n2010-2011")
print("Raw rows:", len(retail_2010_2011))
print("Clean rows:", len(clean_2010_2011))
print("Rows removed:", len(retail_2010_2011) - len(clean_2010_2011))

2009-2010
Raw rows: 525461
Clean rows: 504731
Rows removed: 20730

2010-2011
Raw rows: 541910
Clean rows: 524879
Rows removed: 17031


In [34]:
print("2009-2010 validation")
print("Duplicates:", clean_2009_2010.duplicated().sum())
print("Negative/zero quantity:", (clean_2009_2010["Quantity"] <= 0).sum())
print("Negative/zero price:", (clean_2009_2010["Price"] <= 0).sum())

print("\n2010-2011 validation")
print("Duplicates:", clean_2010_2011.duplicated().sum())
print("Negative/zero quantity:", (clean_2010_2011["Quantity"] <= 0).sum())
print("Negative/zero price:", (clean_2010_2011["Price"] <= 0).sum())

2009-2010 validation
Duplicates: 0
Negative/zero quantity: 0
Negative/zero price: 0

2010-2011 validation
Duplicates: 0
Negative/zero quantity: 0
Negative/zero price: 0


In [35]:
print(
    "2009-2010:",
    clean_2009_2010["InvoiceDate"].min(),
    "to",
    clean_2009_2010["InvoiceDate"].max()
)

print(
    "2010-2011:",
    clean_2010_2011["InvoiceDate"].min(),
    "to",
    clean_2010_2011["InvoiceDate"].max()
)


2009-2010: 2009-12-01 07:45:00 to 2010-12-09 20:01:00
2010-2011: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [36]:
overlap_0910 = clean_2009_2010[
    clean_2009_2010["InvoiceDate"].between(
        "2010-12-01",
        "2010-12-09 23:59:59"
    )
]

overlap_1011 = clean_2010_2011[
    clean_2010_2011["InvoiceDate"].between(
        "2010-12-01",
        "2010-12-09 23:59:59"
    )
]

print("2009-2010 overlap rows:", len(overlap_0910))
print("2010-2011 overlap rows:", len(overlap_1011))

2009-2010 overlap rows: 21696
2010-2011 overlap rows: 21696


In [37]:
overlap_matches = overlap_0910.merge(
    overlap_1011,
    how="inner",
    on=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
)

print("Matching rows across both sheets:", len(overlap_matches))

Matching rows across both sheets: 21696


In [39]:
clean_2010_2011 = clean_2010_2011[
    clean_2010_2011["InvoiceDate"] > "2010-12-09 23:59:59"
].copy()

print(
    clean_2010_2011["InvoiceDate"].min(),
    "to",
    clean_2010_2011["InvoiceDate"].max()
)

retail_clean = pd.concat(
    [clean_2009_2010, clean_2010_2011],
    ignore_index=True
)





2010-12-10 09:33:00 to 2011-12-09 12:50:00


In [40]:
print("Combined rows:", len(retail_clean))
print("Duplicates:", retail_clean.duplicated().sum())
print(
    "Date range:",
    retail_clean["InvoiceDate"].min(),
    "to",
    retail_clean["InvoiceDate"].max()
)

Combined rows: 1007914
Duplicates: 0
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


### Overlapping Date Range

The two source worksheets overlap between 1 December and 9 December 2010.

There were 21,696 cleaned rows in this period in each worksheet, and all 21,696 records matched exactly across both sheets.

To prevent double-counting demand, the duplicated overlap was retained from the first worksheet and removed from the second before concatenation.

In [45]:
retail_clean = retail_clean.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
})

retail_clean.columns

retail_clean["revenue"] = (
    retail_clean["quantity"] * retail_clean["unit_price"]
)

retail_clean[
    ["quantity", "unit_price", "revenue"]
].describe()

,quantity,unit_price,revenue
count,1.007914e+06,1.007914e+06,1.007914e+06
mean,1.111717e+01,4.074618e+00,2.031585e+01
std,1.284700e+02,5.043177e+01,2.057160e+02
min,1.000000e+00,1.000000e-03,1.000000e-03
25%,1.000000e+00,1.250000e+00,4.130000e+00
50%,4.000000e+00,2.100000e+00,1.008000e+01
75%,1.200000e+01,4.130000e+00,1.770000e+01
max,8.099500e+04,2.511109e+04,1.684696e+05


In [46]:
retail_clean.nlargest(
    20,
    "quantity"
)[
    ["invoice", "stock_code", "description",
     "quantity", "unit_price", "revenue",
     "invoice_date", "customer_id", "country"]
]

,invoice,stock_code,description,quantity,unit_price,revenue,invoice_date,customer_id,country
1006439,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446.0,United Kingdom
542684,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346.0,United Kingdom
87083,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.20,2010-02-15 11:57:00,13902.0,Denmark
121864,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.00,2010-03-17 13:09:00,13902.0,Denmark
121866,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.00,2010-03-17 13:09:00,13902.0,Denmark
121867,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.40,2010-03-17 13:09:00,13902.0,Denmark
121865,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.00,2010-03-17 13:09:00,13902.0,Denmark
129396,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom
129397,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom
129398,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.00,2010-03-23 15:36:00,17940.0,United Kingdom


In [47]:
retail_clean.nlargest(
    20,
    "unit_price"
)[
    ["invoice", "stock_code", "description",
     "quantity", "unit_price", "revenue",
     "invoice_date", "customer_id", "country"]
]

,invoice,stock_code,description,quantity,unit_price,revenue,invoice_date,customer_id,country
231524,512771,M,Manual,1,25111.09,25111.09,2010-06-17 16:53:00,NaN,United Kingdom
497491,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33,2010-12-07 15:08:00,NaN,United Kingdom
773541,A563185,B,Adjust bad debt,1,11062.06,11062.06,2011-08-12 14:50:00,NaN,United Kingdom
129383,502263,M,Manual,1,10953.50,10953.50,2010-03-23 15:22:00,12918.0,United Kingdom
129384,502265,M,Manual,1,10953.50,10953.50,2010-03-23 15:28:00,NaN,United Kingdom
328159,522796,M,Manual,1,10468.80,10468.80,2010-09-16 15:12:00,NaN,United Kingdom
344134,524159,M,Manual,1,10468.80,10468.80,2010-09-27 16:12:00,14063.0,United Kingdom
357811,525399,M,Manual,1,10468.80,10468.80,2010-10-05 11:49:00,NaN,United Kingdom
71433,496115,M,Manual,1,8985.60,8985.60,2010-01-29 11:04:00,17949.0,United Kingdom
650644,551697,POST,POSTAGE,1,8142.75,8142.75,2011-05-03 13:46:00,16029.0,United Kingdom


In [48]:
retail_clean.nlargest(
    20,
    "revenue"
)[
    ["invoice", "stock_code", "description",
     "quantity", "unit_price", "revenue",
     "invoice_date", "customer_id", "country"]
]

,invoice,stock_code,description,quantity,unit_price,revenue,invoice_date,customer_id,country
1006439,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,168469.60,2011-12-09 09:15:00,16446.0,United Kingdom
542684,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,77183.60,2011-01-18 10:01:00,12346.0,United Kingdom
698500,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,38970.00,2011-06-10 15:28:00,15098.0,United Kingdom
231524,512771,M,Manual,1,25111.09,25111.09,2010-06-17 16:53:00,NaN,United Kingdom
414828,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,15818.40,2010-11-04 11:36:00,15838.0,United Kingdom
497491,537632,AMAZONFEE,AMAZON FEE,1,13541.33,13541.33,2010-12-07 15:08:00,NaN,United Kingdom
773541,A563185,B,Adjust bad debt,1,11062.06,11062.06,2011-08-12 14:50:00,NaN,United Kingdom
129383,502263,M,Manual,1,10953.50,10953.50,2010-03-23 15:22:00,12918.0,United Kingdom
129384,502265,M,Manual,1,10953.50,10953.50,2010-03-23 15:28:00,NaN,United Kingdom
328159,522796,M,Manual,1,10468.80,10468.80,2010-09-16 15:12:00,NaN,United Kingdom


In [49]:
retail_2010_2011[
    retail_2010_2011["Quantity"].abs() >= 10000
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].sort_values(
    by=["StockCode", "InvoiceDate"]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom


In [50]:
retail_2010_2011[
    retail_2010_2011["StockCode"]
    .astype(str)
    .isin(["23843", "23166"])
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].sort_values("InvoiceDate")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
186770,552882,23166,MEDIUM CERAMIC TOP STORAGE JAR,96,2011-05-12 10:10:00,1.04,14646.0,Netherlands
187196,552953,23166,MEDIUM CERAMIC TOP STORAGE JAR,4,2011-05-12 12:11:00,1.25,16745.0,United Kingdom
187718,553005,23166,MEDIUM CERAMIC TOP STORAGE JAR,5,2011-05-12 16:29:00,1.25,14651.0,United Kingdom
...,...,...,...,...,...,...,...,...
539776,581439,23166,MEDIUM CERAMIC TOP STORAGE JAR,2,2011-12-08 16:30:00,2.46,NaN,United Kingdom
540301,581476,23166,MEDIUM CERAMIC TOP STORAGE JAR,48,2011-12-09 08:48:00,1.04,12433.0,Norway
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


In [51]:
raw_2010_2011_no_overlap = retail_2010_2011[
    retail_2010_2011["InvoiceDate"] > "2010-12-09 23:59:59"
].copy()

retail_raw_combined = pd.concat(
    [retail_2009_2010, raw_2010_2011_no_overlap],
    ignore_index=True
)

positive_sales = retail_raw_combined[
    retail_raw_combined["Quantity"] > 0
].copy()

cancellations = retail_raw_combined[
    retail_raw_combined["Quantity"] < 0
].copy()

cancellations["cancel_quantity"] = cancellations["Quantity"].abs()

In [54]:
candidate_matches = cancellations.merge(
    positive_sales,
    left_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "cancel_quantity"
    ],
    right_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "Quantity"
    ],
    suffixes=("_cancel", "_sale")
)

candidate_matches = candidate_matches[
    candidate_matches["InvoiceDate_sale"]
    <= candidate_matches["InvoiceDate_cancel"]
].copy()

candidate_matches["time_gap"] = (
    candidate_matches["InvoiceDate_cancel"]
    - candidate_matches["InvoiceDate_sale"]
)

candidate_matches[
    [
        "Invoice_sale",
        "Invoice_cancel",
        "StockCode",
        "Quantity_sale",
        "Price",
        "Customer ID",
        "InvoiceDate_sale",
        "InvoiceDate_cancel",
        "time_gap"
    ]
].sort_values("time_gap").head(30)

,Invoice_sale,Invoice_cancel,StockCode,Quantity_sale,Price,Customer ID,InvoiceDate_sale,InvoiceDate_cancel,time_gap
2900,508382,C508384,20685,2,7.49,13634.0,2010-05-14 14:18:00,2010-05-14 14:18:00,0 days
2879,508382,C508384,47590A,3,5.45,13634.0,2010-05-14 14:18:00,2010-05-14 14:18:00,0 days
2950,508382,C508384,84879,16,1.69,13634.0,2010-05-14 14:18:00,2010-05-14 14:18:00,0 days
4085,514874,C514873,22423,16,10.95,14154.0,2010-07-07 10:26:00,2010-07-07 10:26:00,0 days
514,493487,C493488,72008,24,0.42,NaN,2010-01-04 14:50:00,2010-01-04 14:50:00,0 days
4676,518543,C518544,22151,24,0.42,17314.0,2010-08-09 18:20:00,2010-08-09 18:20:00,0 days
4678,518543,C518544,22535,48,0.42,17314.0,2010-08-09 18:20:00,2010-08-09 18:20:00,0 days
4680,518543,C518544,22534,48,0.42,17314.0,2010-08-09 18:20:00,2010-08-09 18:20:00,0 days
4682,518543,C518544,22139,3,4.95,17314.0,2010-08-09 18:20:00,2010-08-09 18:20:00,0 days
4684,518543,C518544,10002,12,0.85,17314.0,2010-08-09 18:20:00,2010-08-09 18:20:00,0 days


In [56]:
retail_raw_combined = retail_raw_combined.reset_index(drop=True)
retail_raw_combined["row_id"] = retail_raw_combined.index

positive_sales = retail_raw_combined[
    retail_raw_combined["Quantity"] > 0
].copy()

cancellations = retail_raw_combined[
    retail_raw_combined["Quantity"] < 0
].copy()

cancellations["cancel_quantity"] = cancellations["Quantity"].abs()

candidate_matches = cancellations.merge(
    positive_sales,
    left_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "cancel_quantity"
    ],
    right_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "Quantity"
    ],
    suffixes=("_cancel", "_sale")
)

candidate_matches = candidate_matches[
    candidate_matches["InvoiceDate_sale"]
    <= candidate_matches["InvoiceDate_cancel"]
].copy()

candidate_matches["time_gap"] = (
    candidate_matches["InvoiceDate_cancel"]
    - candidate_matches["InvoiceDate_sale"]
)

total_cancellations = len(cancellations)

matched_cancellations = (
    candidate_matches["row_id_cancel"].nunique()
)

print("Total negative transactions:", total_cancellations)
print("Cancellations with candidate match:", matched_cancellations)
print(
    "No matching positive transaction:",
    total_cancellations - matched_cancellations
)

print(
    "Match rate:",
    round(matched_cancellations / total_cancellations * 100, 2),
    "%"
)

Total negative transactions: 22557
Cancellations with candidate match: 6630
No matching positive transaction: 15927
Match rate: 29.39 %


In [57]:
match_counts = (
    candidate_matches
    .groupby("row_id_cancel")
    .size()
)

print("Cancellations with one candidate:", (match_counts == 1).sum())
print("Cancellations with multiple candidates:", (match_counts > 1).sum())

Cancellations with one candidate: 4983
Cancellations with multiple candidates: 1647


In [58]:
best_matches = (
    candidate_matches
    .sort_values(
        ["row_id_cancel", "time_gap"],
        ascending=[True, True]
    )
    .drop_duplicates(
        subset="row_id_cancel",
        keep="first"
    )
)

print("Matched cancellations:", len(best_matches))

print(
    "Unique positive sales being reversed:",
    best_matches["row_id_sale"].nunique()
)

print(
    "Duplicate use of positive sales:",
    len(best_matches) - best_matches["row_id_sale"].nunique()
)

Matched cancellations: 6630
Unique positive sales being reversed: 6461
Duplicate use of positive sales: 169


In [69]:
retail_raw_combined["invoice_day"] = (
    retail_raw_combined["InvoiceDate"].dt.date
)

positive_rows = retail_raw_combined[
    retail_raw_combined["Quantity"] > 0
].copy()

negative_rows = retail_raw_combined[
    retail_raw_combined["Quantity"] < 0
].copy()

negative_rows["match_quantity"] = negative_rows["Quantity"].abs()
positive_rows["match_quantity"] = positive_rows["Quantity"]

same_day_reversals = negative_rows.merge(
    positive_rows,
    left_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "match_quantity",
        "invoice_day"
    ],
    right_on=[
        "StockCode",
        "Customer ID",
        "Price",
        "match_quantity",
        "invoice_day"
    ],
    suffixes=("_cancel", "_sale")
)

same_day_reversals = same_day_reversals[
    same_day_reversals["InvoiceDate_sale"]
    <= same_day_reversals["InvoiceDate_cancel"]
].copy()

print("Same-day reversal matches:", len(same_day_reversals))

same_day_reversals[
    [
        "Invoice_sale",
        "Invoice_cancel",
        "StockCode",
        "match_quantity",
        "Price",
        "Customer ID",
        "InvoiceDate_sale",
        "InvoiceDate_cancel"
    ]
].sort_values(
    "match_quantity",
    ascending=False
).head(20)

Same-day reversal matches: 1593


,Invoice_sale,Invoice_cancel,StockCode,match_quantity,Price,Customer ID,InvoiceDate_sale,InvoiceDate_cancel
2265,581483,C581484,23843,80995,2.08,16446.0,2011-12-09 09:15:00,2011-12-09 09:27:00
1420,541431,C541433,23166,74215,1.04,12346.0,2011-01-18 10:01:00,2011-01-18 10:17:00
397,504453,504455,22528,4800,0.00,NaN,2010-04-13 15:34:00,2010-04-13 15:34:00
398,504456,504457,22558,4800,0.00,NaN,2010-04-13 15:35:00,2010-04-13 15:35:00
777,516039,516127,22643,2560,0.00,NaN,2010-07-16 10:54:00,2010-07-16 15:00:00
774,516038,516126,22642,2560,0.00,NaN,2010-07-16 10:54:00,2010-07-16 15:00:00
772,516034,516122,22638,2560,0.00,NaN,2010-07-16 10:52:00,2010-07-16 14:59:00
775,516036,516124,22640,2560,0.00,NaN,2010-07-16 10:53:00,2010-07-16 15:00:00
776,516037,516125,22641,2560,0.00,NaN,2010-07-16 10:53:00,2010-07-16 15:00:00
773,516035,516123,22639,2560,0.00,NaN,2010-07-16 10:52:00,2010-07-16 15:00:00


In [70]:
retail_raw_combined = retail_raw_combined.reset_index(drop=True)
retail_raw_combined["row_id"] = retail_raw_combined.index

retail_raw_combined["invoice_day"] = (
    retail_raw_combined["InvoiceDate"].dt.date
)

positive_rows = retail_raw_combined[
    (retail_raw_combined["Quantity"] > 0)
    & (retail_raw_combined["Price"] > 0)
].copy()

cancel_rows = retail_raw_combined[
    (retail_raw_combined["Quantity"] < 0)
    & (retail_raw_combined["Price"] > 0)
    & (
        retail_raw_combined["Invoice"]
        .astype(str)
        .str.startswith("C")
    )
].copy()

positive_rows["match_quantity"] = positive_rows["Quantity"]

cancel_rows["match_quantity"] = (
    cancel_rows["Quantity"].abs()
)

confirmed_reversals = cancel_rows.merge(
    positive_rows,
    on=[
        "StockCode",
        "Customer ID",
        "Price",
        "match_quantity",
        "invoice_day"
    ],
    suffixes=("_cancel", "_sale")
)

confirmed_reversals = confirmed_reversals[
    confirmed_reversals["InvoiceDate_sale"]
    <= confirmed_reversals["InvoiceDate_cancel"]
].copy()

print("Confirmed reversal matches:", len(confirmed_reversals))

print(
    "Unique positive sales reversed:",
    confirmed_reversals["row_id_sale"].nunique()
)

Confirmed reversal matches: 1563
Unique positive sales reversed: 1491


In [72]:
reversed_sale_ids = (
    confirmed_reversals["row_id_sale"]
    .drop_duplicates()
)

len(reversed_sale_ids)

retail_final = retail_raw_combined.copy()

retail_final = retail_final[
    (retail_final["Quantity"] > 0)
    & (retail_final["Price"] > 0)
    & (~retail_final["row_id"].isin(reversed_sale_ids))
].copy()

retail_final = retail_final.drop_duplicates(
    subset=[
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
)

retail_final = retail_final.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
})

retail_final["revenue"] = (
    retail_final["quantity"]
    * retail_final["unit_price"]
)

retail_final = retail_final.drop(
    columns=["row_id", "invoice_day"],
    errors="ignore"
)

print("Final rows:", len(retail_final))
print("Duplicates:", retail_final.duplicated().sum())
print("Non-positive quantity:", (retail_final["quantity"] <= 0).sum())
print("Non-positive price:", (retail_final["unit_price"] <= 0).sum())

print(
    "Date range:",
    retail_final["invoice_date"].min(),
    "to",
    retail_final["invoice_date"].max()
)

Final rows: 1006465
Duplicates: 0
Non-positive quantity: 0
Non-positive price: 0
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [73]:
retail_final.nlargest(
    10,
    "quantity"
)[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "revenue"
    ]
]

,invoice,stock_code,description,quantity,unit_price,revenue
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,1915.2
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,1296.0
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,1296.0
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,1274.4
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,1248.0
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,2500.0
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,2500.0
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,2500.0
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,2500.0
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,0.30,2836.8


In [74]:
top_quantity_rows = retail_final.nlargest(10, "quantity")

top_quantity_rows[
    [
        "invoice",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "customer_id",
        "country",
        "invoice_date"
    ]
]

,invoice,stock_code,description,quantity,unit_price,customer_id,country,invoice_date
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0,Denmark,2010-02-15 11:57:00
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0,Denmark,2010-03-17 13:09:00
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0,Denmark,2010-03-17 13:09:00
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0,Denmark,2010-03-17 13:09:00
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0,Denmark,2010-03-17 13:09:00
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0,United Kingdom,2010-03-23 15:36:00
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,17940.0,United Kingdom,2010-03-23 15:36:00
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,17940.0,United Kingdom,2010-03-23 15:36:00
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,0.25,17940.0,United Kingdom,2010-03-23 15:36:00
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,0.30,13902.0,Denmark,2010-02-17 10:51:00


In [75]:
top_stock_codes = (
    top_quantity_rows["stock_code"]
    .astype(str)
    .tolist()
)

retail_raw_combined[
    retail_raw_combined["StockCode"]
    .astype(str)
    .isin(top_stock_codes)
][
    [
        "Invoice",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "Price",
        "Customer ID",
        "Country"
    ]
].sort_values(
    ["StockCode", "InvoiceDate"]
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
142,489445,21085,SET/6 WOODLAND PAPER CUPS,24,2009-12-01 09:57:00,0.65,17519.0,United Kingdom
7875,490074,21085,SET/6 WOODLAND PAPER CUPS,1,2009-12-03 14:39:00,1.70,NaN,United Kingdom
13602,490490,21085,SET/6 WOODLAND PAPER CUPS,2,2009-12-06 13:10:00,0.65,14546.0,United Kingdom
19193,490964,21085,SET/6 WOODLAND PAPER CUPS,1,2009-12-08 16:09:00,0.65,14646.0,Netherlands
41953,492830,21085,SET/6 WOODLAND PAPER CUPS,24,2009-12-20 15:38:00,0.65,12683.0,France
...,...,...,...,...,...,...,...,...
56480,494473,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,24,2010-01-14 14:53:00,1.65,17508.0,Greece
90860,497946,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,2504,2010-02-15 11:57:00,1.45,13902.0,Denmark
93676,C498151,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,-2504,2010-02-17 10:37:00,1.45,13902.0,Denmark
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902.0,Denmark


In [76]:
print("Final rows:", len(retail_final))
print("Duplicates:", retail_final.duplicated().sum())
print("Non-positive quantity:", (retail_final["quantity"] <= 0).sum())
print("Non-positive price:", (retail_final["unit_price"] <= 0).sum())

Final rows: 1006465
Duplicates: 0
Non-positive quantity: 0
Non-positive price: 0


In [77]:
non_product_codes = [
    "M",
    "AMAZONFEE",
    "B",
    "DOT",
    "D"
]

retail_final[
    retail_final["stock_code"]
    .astype(str)
    .isin(non_product_codes)
][
    ["stock_code", "description"]
].value_counts()

stock_code  description    
DOT         DOTCOM POSTAGE     1415
M           Manual              781
D           Discount              5
AMAZONFEE   AMAZON FEE            1
B           Adjust bad debt       1
Name: count, dtype: int64

In [78]:
non_product_codes = [
    "DOT",
    "M",
    "D",
    "AMAZONFEE",
    "B"
]

retail_final = retail_final[
    ~retail_final["stock_code"]
    .astype(str)
    .isin(non_product_codes)
].copy()

In [80]:
print("Final rows:", len(retail_final))
print("Duplicates:", retail_final.duplicated().sum())
print("Products:", retail_final["stock_code"].nunique())
print("Customers:", retail_final["customer_id"].nunique())
print("Countries:", retail_final["country"].nunique())

print(
    "Date range:",
    retail_final["invoice_date"].min(),
    "to",
    retail_final["invoice_date"].max()
)

retail_final.to_csv(
    "../data/cleaned/retail_transactions_clean.csv",
    index=False
)

Final rows: 1004262
Duplicates: 0
Products: 4907
Customers: 5861
Countries: 43
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
